# Lab 5 - Silver layer design review

## Q1. Quarantine versus delete

Bad orders are kept in `quarantine_orders.csv` instead of being deleted because the file is **evidence, not garbage**. A rejected row usually points to a real upstream problem - a mistyped `customer_id`, a broken API feed, a wrong date format in one source system. If we delete it, we lose the only trace that the problem exists and can never repair and reprocess the record. Keeping it also gives management a measurable number (65 orders, about 2 percent of the week) instead of an invisible loss.

In a real company the **data steward / data quality analyst** owns this file, sometimes together with the operational team that owns the source system. They review it on a schedule, contact the system owner, fix the cause at the source, and re-run the pipeline so the rows can return to the silver layer. Deleting the rows would silently drop revenue from analytics and hide the bug.

## Q2. Hash versus mask

`email_hash` is for **analysis**: it is stable and unique, so analysts can join tables and count unique customers (`GROUP BY email_hash`) without ever seeing an address. `email_masked` is for **display**: a support agent can recognise which customer replied but cannot copy the full address or contact them.

Hashing **without a salt** is weak for emails because the input space is tiny and guessable. An attacker who steals the table can simply hash a dictionary of candidate addresses (john at gmail, ali at yahoo, and so on) and look for matches. No brute force over 2^256 is needed, just a list of a few million realistic emails. The salt is a secret string mixed in before hashing, so the attacker's candidate list produces completely different hashes unless they also steal the salt from `.env`. The salt moves the security of the pseudonym from being hard to guess the input to being hard to steal the secret.


## Q3. Data minimisation

Name, phone and MyKad never reach silver. This follows the **General Principle of the PDPA 2010**: personal data may only be processed for a lawful purpose directly related to the activity, and the data collected must be adequate but not excessive for that purpose, which is data minimisation. Dropping the direct identifiers also supports the **Retention Principle**, since data must not be kept longer than needed for the purpose and identifying a real person is no longer needed once analytics only needs segments.

`age_group` is safer than `birth_year` because a birth year is much closer to an identifier. Combined with `state` and `signup_date`, a birth year narrows a person down to a small group and makes re-identification realistic through a linkage attack. A ten-year band such as 25-34 raises the size of the anonymity group enormously, so an individual cannot be singled out, while the marketing insight of which age band buys most is still available.


## Q4. Parquet versus CSV

CSV stores every value as text, so a reader has to guess the types back. Parquet stores the **schema with the data**: `order_date` stays `datetime64[us]`, `quantity` stays `int64`, `in_stock`(in books) stays `bool`, and identifier columns stay strings. That matters here: `order_id` looks like `000017`, and a CSV reader that infers integers would turn it into `17` and destroy the leading zeros. Parquet also keeps nulls as real nulls, and reads only the columns you ask for (columnar and compressed), which is why the silver layer is stored this way.


In [2]:
import pandas as pd

orders = pd.read_parquet("../data/lake/silver/orders.parquet")
print(orders.head())
print()
print(orders.dtypes)
print()
print("rows:", len(orders))


  order_id customer_id                           book_id  quantity order_date
0   O00001       C0141                   the-republic_78         2 2026-03-02
1   O00002       C0344     the-nanny-diaries-nanny-1_265         2 2026-05-10
2   O00003       C0030               i-will-find-you_524         1 2026-05-28
3   O00004       C0262                     gratitude_163         2 2026-03-02
4   O00005       C0279  kitchens-of-the-great-midwest_28         1 2026-07-26

order_id               object
customer_id            object
book_id                object
quantity                int64
order_date     datetime64[us]
dtype: object

rows: 2935
